[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Open-Athena/MarinFold/blob/main/notebooks/short_document_bias.ipynb)

# Does contacts-v1 generate "too-short" documents? — MarinFold [#142](https://github.com/Open-Athena/MarinFold/issues/142)

contacts-v1 predicts residue–residue contacts by **generating a document** of `<contact> <pI> <pJ>`
statements from the sequence alone. A natural worry: does the model **stop early** and emit **far
fewer contacts than the structure has**, capping recall no matter how good the inference wrapper is?

This notebook **generates rollouts** from the model for a handful of eval proteins and **analyzes**
them: how many contacts each document asserts vs the ground-truth count (`pred/gt`), whether the
documents finish on their own (`<end>`) or get truncated, and how that relates to accuracy. It then
**compares sampling recipes** — including turning top-k / top-p off — to see whether the shortfall is
a decoding artifact.

**What to look for.** `pred/gt` lands a bit under 1 (mild-to-moderate under-generation), **100%
finished** (the model chooses `<end>` — a generous token budget never truncates), and the shortfall is
**biggest exactly where recall is worst** — i.e. short documents are a *symptom* of the model being
unsure of the fold, strong only on hard/long proteins, rather than a decoding bug.

Runs on a GPU — a Colab **T4** is enough. Model: the exp75 E8 1.5B (`step-35679`, eval loss 2.7566);
switchable to the newer default `contacts-v1-exp120-1.5B`.

In [ ]:
# @title Configuration
MODEL_NICKNAME = "contacts-v1-exp75-1.5B"  # @param ["contacts-v1-exp75-1.5B", "contacts-v1-exp120-1.5B"]
# A short / confident-long / lost-long trio spans the length range and shows the symptom contrast.
# Add any of the 12 eval entry_ids (see the targets table printed below).
PROTEINS = "denovo_pdb__1uw1_A, denovo_pdb__6reo_A, foldbench100__8bfy_A"  # @param {type:"string"}
N_ROLLOUTS = 60  # @param {type:"integer"}
TEMPERATURE = 1.0  # @param {type:"number"}
TOP_P = 0.95  # @param {type:"number"}
TOP_K = 50  # @param {type:"integer"}
print(dict(model=MODEL_NICKNAME, proteins=[x.strip() for x in PROTEINS.split(",")],
           n_rollouts=N_ROLLOUTS, temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K))

In [ ]:
# Setup: clone MarinFold + install marinfold[transformers] (build_document + model registry),
# and fetch the eval proteins (sequence + resolved ground-truth contacts). marinfold pins
# transformers<5 so the Levanter rope config loads correctly.
import importlib, importlib.util, io, subprocess, sys, urllib.request
from pathlib import Path
import numpy as np, pandas as pd, pyarrow.parquet as pq

REPO = Path("/content/MarinFold") if Path("/content").exists() else Path.cwd() / "MarinFold"
if not (REPO / "marinfold" / "pyproject.toml").exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/Open-Athena/MarinFold.git", str(REPO)], check=True)
if importlib.util.find_spec("marinfold") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-e", f"{REPO / 'marinfold'}[transformers]"], check=True)
    if str(REPO / "marinfold") not in sys.path:
        sys.path.insert(0, str(REPO / "marinfold"))
    importlib.invalidate_caches()

RAW = "https://raw.githubusercontent.com/Open-Athena/MarinFold/06014e29c1f08cd7adea29f9d821ec0ac829c6c8/experiments/exp142_evals_short_document_bias"
def fetch_parquet(name):
    with urllib.request.urlopen(f"{RAW}/{name}") as r:
        return pq.read_table(io.BytesIO(r.read())).to_pandas()

targets = fetch_parquet("targets.parquet").set_index("entry_id")
print("marinfold ready; eval proteins:")
print(targets.reset_index()[["entry_id", "dataset", "L", "n_gt"]].sort_values("L").to_string(index=False))

## Generate rollouts

For each protein we build `N_ROLLOUTS` fresh contacts-v1 document realizations (resampled N-terminus
start + statement order — the format's nuisance symmetries), sample a completion of the contact
section, and record per rollout: contacts emitted (`n_pred`, deduped, sep≥6), whether it finished
(`<end>`), the document length in tokens, and recall/precision against the ground truth. The token
budget is deliberately generous (`min(8192−prefix, 6·L+128)`) so a short document can only be the
model's own choice.

In [ ]:
import re, time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from marinfold.registry import resolve_model
from marinfold.document_structures.contacts_v1 import (
    GenerationConfig, build_document, residues_from_sequence,
)

BEGIN, NUM_POS, MIN_SEP = "<begin_statements>", 2000, 6
CONTACT_RE = re.compile(r"<contact>\s+<p(\d+)>\s+<p(\d+)>")

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = (torch.bfloat16 if device == "cuda" and torch.cuda.is_bf16_supported()
         else torch.float16 if device == "cuda" else torch.float32)

model_dir = resolve_model(MODEL_NICKNAME)          # downloads from the public bucket (anonymous)
tok = AutoTokenizer.from_pretrained(str(model_dir))
if tok.pad_token_id is None:
    tok.pad_token = "<end>"
model = AutoModelForCausalLM.from_pretrained(str(model_dir), dtype=dtype).to(device).eval()
END = tok.convert_tokens_to_ids("<end>")
CONTACT_ID = tok.convert_tokens_to_ids("<contact>")
print(f"loaded {MODEL_NICKNAME} on {device} ({dtype}); vocab={model.config.vocab_size}")


def prefix_and_map(entry, seq, r):
    """One fresh contacts-v1 realization (resampled N-term + statement order)."""
    doc = build_document(f"{entry}:r{r}", residues_from_sequence(seq), [], config=GenerationConfig())
    prefix = doc.document[: doc.document.index(BEGIN) + len(BEGIN)]
    pos2seq = {(doc.n_term_index + i) % NUM_POS: i for i in range(doc.seq_len)}
    return prefix, pos2seq


@torch.no_grad()
def rollouts_for(entry, n, temperature=None, top_p=None, top_k=None,
                 batch=16 if torch.cuda.is_available() else 4):
    """Sample `n` rollouts for one protein. Sampling knobs default to the config above;
    pass them explicitly to compare recipes (top_p=1.0 + top_k=0 == no truncation)."""
    temperature = TEMPERATURE if temperature is None else temperature
    top_p = TOP_P if top_p is None else top_p
    top_k = TOP_K if top_k is None else top_k
    row = targets.loc[entry]
    seq, L, n_gt = row["sequence"], int(row["L"]), int(row["n_gt"])
    gt = {(int(i), int(j)) for i, j in row["gt_contacts"]}
    built = [prefix_and_map(entry, seq, r) for r in range(n)]
    plen = len(tok(built[0][0], add_special_tokens=False).input_ids)
    max_new = min(8192 - plen, 6 * L + 128)        # generous: the cap never binds
    recs = []
    for s in range(0, n, batch):
        chunk = built[s:s + batch]
        enc = tok([c[0] for c in chunk], add_special_tokens=False,
                  return_tensors="pt", padding=True).to(device)
        out = model.generate(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"],
                             do_sample=True, temperature=temperature, top_p=top_p, top_k=top_k,
                             max_new_tokens=max_new, eos_token_id=END, pad_token_id=END)
        for (prefix, pos2seq), g in zip(chunk, out[:, enc["input_ids"].shape[1]:].tolist()):
            finished = END in g
            g = g[: g.index(END) + 1] if finished else g
            raw_stmts = sum(1 for t in g if t == CONTACT_ID)
            pred = set()
            for a, b in CONTACT_RE.findall(tok.decode(g, skip_special_tokens=False)):
                ia, ib = pos2seq.get(int(a)), pos2seq.get(int(b))
                if ia is None or ib is None or abs(ia - ib) < MIN_SEP:
                    continue
                pred.add((min(ia, ib), max(ia, ib)))
            tp = len(pred & gt)
            recs.append(dict(entry=entry, L=L, n_gt=n_gt, n_pred=len(pred), raw_stmts=raw_stmts,
                             finished=finished, n_gen_tokens=len(g),
                             recall=tp / len(gt) if gt else np.nan,
                             precision=tp / len(pred) if pred else 0.0))
    return recs


def add_ratios(df):
    df = df.copy()
    df["pred_gt"] = df.n_pred / df.n_gt
    p, r = df.precision, df.recall
    df["f1"] = np.where((p + r) > 0, 2 * p * r / (p + r), 0.0)
    return df


chosen = [p.strip() for p in PROTEINS.split(",") if p.strip()]
rows = []
for e in chosen:
    t0 = time.time()
    rs = rollouts_for(e, N_ROLLOUTS)
    rows += rs
    pg = np.mean([x["n_pred"] for x in rs]) / rs[0]["n_gt"]
    print(f"  {e:<24} L={rs[0]['L']:>3} n_gt={rs[0]['n_gt']:>3}  pred/gt={pg:.2f}  "
          f"finished={np.mean([x['finished'] for x in rs]):.2f}  "
          f"recall={np.nanmean([x['recall'] for x in rs]):.2f}  ({time.time()-t0:.0f}s)")
roll = add_ratios(pd.DataFrame(rows))

## Analysis

In [ ]:
agg = (roll.groupby(["entry", "L", "n_gt"])
           .agg(rollouts=("n_pred", "size"), emitted=("n_pred", "mean"), pred_gt=("pred_gt", "mean"),
                frac_lt_half=("pred_gt", lambda s: (s < 0.5).mean()), tokens=("n_gen_tokens", "mean"),
                finished=("finished", "mean"), recall=("recall", "mean"),
                precision=("precision", "mean"), best_f1=("f1", "max"))
           .reset_index().sort_values("L"))
pd.set_option("display.width", 160)
print(agg.round(3).to_string(index=False))

print(f"\npooled over {len(roll)} rollouts:  pred/gt mean {roll.pred_gt.mean():.2f} "
      f"(median {roll.pred_gt.median():.2f})   finished {100*roll.finished.mean():.0f}%   "
      f"rollouts emitting < half the GT count {100*(roll.pred_gt < 0.5).mean():.0f}%")
if len(agg) >= 3:
    print(f"corr(pred/gt, recall) across proteins = {np.corrcoef(agg.pred_gt, agg.recall)[0,1]:+.2f} "
          "  (positive => under-generation tracks accuracy, i.e. a symptom not a decoding bug)")

In [ ]:
import matplotlib.pyplot as plt

order = agg.entry.tolist()
Ls = agg.L.values
fig, ax = plt.subplots(2, 2, figsize=(13, 9))
wid = max(6, (Ls.max() - Ls.min()) / 25) if len(Ls) > 1 else 8

# (1) pred/gt distribution per protein vs L
groups = [roll.loc[roll.entry == e, "pred_gt"].values for e in order]
bp = ax[0,0].boxplot(groups, positions=Ls, widths=wid, showfliers=False,
                     patch_artist=True, manage_ticks=False)
for b in bp["boxes"]: b.set(facecolor="#4C72B0", alpha=0.5)
ax[0,0].axhline(1.0, color="crimson", ls="--", lw=1.5, label="parity (pred = GT count)")
ax[0,0].axhline(0.5, color="gray", ls=":", label="half of GT")
ax[0,0].set(xlabel="protein length L", ylabel="contacts emitted / n_gt", title="Contacts emitted vs ground truth")
ax[0,0].set_ylim(0, max(1.6, min(3.0, roll.pred_gt.max()))); ax[0,0].legend(fontsize=8)

# (2) mean emitted vs n_gt (parity)
sc = ax[0,1].scatter(agg.n_gt, agg.emitted, c=Ls, cmap="viridis", s=80, zorder=3)
mx = max(agg.n_gt.max(), agg.emitted.max()) * 1.05
ax[0,1].plot([0, mx], [0, mx], "crimson", ls="--", label="y = x (parity)")
for _, row in agg.iterrows():
    ax[0,1].annotate(f"L{int(row.L)}", (row.n_gt, row.emitted), fontsize=8, xytext=(4,4), textcoords="offset points")
ax[0,1].set(xlabel="n_gt (ground-truth contacts)", ylabel="mean contacts emitted", title="Mean contacts emitted vs GT count")
ax[0,1].legend(fontsize=8); fig.colorbar(sc, ax=ax[0,1], label="L")

# (3) recall & precision vs L
ax[1,0].plot(Ls, agg.recall, "o-", color="#4C72B0", label="recall (all, sep>=6)")
ax[1,0].plot(Ls, agg.precision, "^--", color="#C44E52", label="precision (all)")
ax[1,0].set(xlabel="protein length L", ylabel="mean per-rollout metric", title="Rollout accuracy vs length", ylim=(0, None))
ax[1,0].legend(fontsize=8)

# (4) generated tokens vs complete-doc tokens
full = 3 * agg.n_gt + 1
ax[1,1].scatter(full, agg.tokens, c=Ls, cmap="viridis", s=80, zorder=3)
mx = max(full.max(), agg.tokens.max()) * 1.05
ax[1,1].plot([0, mx], [0, mx], "crimson", ls="--", label="y = x (complete doc)")
ax[1,1].set(xlabel="tokens for a complete GT doc (3*n_gt+1)", ylabel="mean generated tokens", title="Document length: generated vs complete")
ax[1,1].legend(fontsize=8)

fig.suptitle(f"{MODEL_NICKNAME} - short-document-bias probe (your {N_ROLLOUTS} rollouts/protein)", fontsize=12)
fig.tight_layout(); plt.show()

## Compare sampling recipes — is the shortfall just top-k / top-p?

Truncated sampling renormalizes over the kept tokens, which **boosts every kept token — including
`<end>`** — so top-k / top-p can make the model stop earlier. Does turning them off fix the short
documents? This section reruns the same generation under several recipes and compares them.

Watch two things together: **`pred_gt`** (does the count shortfall go away?) and **`recall` /
`precision` / `f1`** (does it actually help?). The informative outcome is when the count moves a lot
and the accuracy doesn't — that means the extra statements are low-confidence guesses, i.e. you've
traded under-generation for noise rather than fixing anything.

⏱️ Runtime scales with `recipes × proteins × rollouts`, and untruncated sampling makes documents
longer (slower). On a T4, start with the two default recipes; drop the longest protein or lower
`COMPARE_N_ROLLOUTS` if it drags.

In [ ]:
# @title Recipe comparison settings
COMPARE_N_ROLLOUTS = 30  # @param {type:"integer"}

# name -> (temperature, top_p, top_k).  top_p=1.0 AND top_k=0 == no truncation
# (sample the model's raw distribution). Add/remove freely.
RECIPES = {
    "standard (p0.95,k50)": (1.0, 0.95, 50),
    "no trunc (p1.0,k0)":   (1.0, 1.00, 0),
    # Uncomment to isolate which knob matters (each adds a full pass):
    # "no top-k (p0.95,k0)":  (1.0, 0.95, 0),
    # "no top-p (p1.0,k50)":  (1.0, 1.00, 50),
    # Temperature is a separate knob you can probe the same way:
    # "cooler   (T0.7)":      (0.7, 0.95, 50),
}

rows = []
for name, (temp, tp, tk) in RECIPES.items():
    for e in chosen:
        t0 = time.time()
        rs = rollouts_for(e, COMPARE_N_ROLLOUTS, temperature=temp, top_p=tp, top_k=tk)
        for x in rs:
            x["recipe"] = name
        rows += rs
        pg = np.mean([x["n_pred"] for x in rs]) / rs[0]["n_gt"]
        print(f"  {name:<22} {e.split('__')[-1]:<9} L={rs[0]['L']:>3}  pred/gt={pg:.2f}  "
              f"recall={np.nanmean([x['recall'] for x in rs]):.3f}  ({time.time()-t0:.0f}s)")
cmp_roll = add_ratios(pd.DataFrame(rows))

In [ ]:
cmp_agg = (cmp_roll.groupby(["entry", "L", "n_gt", "recipe"])
               .agg(pred_gt=("pred_gt", "mean"), tokens=("n_gen_tokens", "mean"),
                    finished=("finished", "mean"), recall=("recall", "mean"),
                    precision=("precision", "mean"), f1=("f1", "mean"))
               .reset_index().sort_values(["L", "recipe"]))
print("per protein x recipe:")
print(cmp_agg.round(3).to_string(index=False))

print("\npooled across proteins:")
print(cmp_roll.groupby("recipe")
              .agg(pred_gt=("pred_gt", "mean"), tokens=("n_gen_tokens", "mean"),
                   finished=("finished", "mean"), recall=("recall", "mean"),
                   precision=("precision", "mean"), f1=("f1", "mean"))
              .round(3).to_string())

In [ ]:
ent = cmp_agg[["entry", "L"]].drop_duplicates().sort_values("L")
entries = ent.entry.tolist()
labels = [f"{e.split('__')[-1]}\nL={int(l)}" for e, l in ent.values]
recipes = list(RECIPES)
x = np.arange(len(entries)); w = 0.8 / max(len(recipes), 1)

panels = [("pred_gt", "contacts emitted / n_gt", 1.0), ("recall", "recall (all)", None),
          ("precision", "precision (all)", None), ("f1", "F1 (all)", None)]
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, (key, title, ref) in zip(axes.ravel(), panels):
    for i, rc in enumerate(recipes):
        vals = [cmp_agg[(cmp_agg.entry == e) & (cmp_agg.recipe == rc)][key].iloc[0] for e in entries]
        ax.bar(x + i * w - 0.4 + w / 2, vals, w, label=rc)
    if ref is not None:
        ax.axhline(ref, color="crimson", ls="--", lw=1.2, label="parity")
    ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=8)
    ax.set_title(title); ax.set_ylabel(title); ax.grid(axis="y", alpha=0.3)
axes[0, 0].legend(fontsize=7)
fig.suptitle("Sampling recipes — does disabling truncation fix the short-document bias?", fontsize=12)
fig.tight_layout(); plt.show()

## Takeaway

**The short documents are real, but they're a symptom.** Contacts emitted sit below parity, biggest
exactly where recall is worst, and with a 100% finish rate that shortfall is the model choosing
`<end>` — not a truncated generation.

**Turning off top-k / top-p lengthens the documents but doesn't help.** In the recipe comparison you
should see `pred_gt` climb toward (and on hard/long proteins past) parity while `recall` barely moves
and `precision`/`f1` *fall* — the extra statements come from the low-probability tail, so they're
valid-but-wrong contacts. Truncation genuinely was shortening documents (it renormalizes `<end>`
upward), but removing it just trades under-generation for noise.

So neither decoding knob is the lever: forcing more contacts doesn't make the model *know* more
contacts. The fix is a model that knows the fold — matching exp82's finding that better inference
narrows but doesn't close the gap.

> Note: these are **per-rollout** metrics. The settled inference recipe ([exp82](https://github.com/Open-Athena/MarinFold/issues/82))
> instead *votes across ~100 rollouts* and ranks pairs by frequency, where extra sampling diversity
> could trade off differently — a separate experiment.

- Issue & full writeup: [MarinFold #142](https://github.com/Open-Athena/MarinFold/issues/142)
- Code & offline analysis: [`experiments/exp142_evals_short_document_bias/`](https://github.com/Open-Athena/MarinFold/tree/main/experiments/exp142_evals_short_document_bias)